In [1]:
import random
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from models.ngram.knn import KNN
from models.ngram.model import Model
from collections import Counter, defaultdict

load_dotenv(Path("../.env"))

/home/bugslayer/Documents/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="note_table")
loader = dataloader.loader(load_music=dataloader.load_note_table_music)

In [3]:
tokens_by_pitch = defaultdict(Counter)

for batch in loader:
    for song in batch:
        note_tokens = zip(
            song["pitch"],
            song["velocity_bin"],
            song["delta_onset_bin"],
            song["duration_bin"],
        )

        for token in note_tokens:
            tokens_by_pitch[token[0]][token] += 1

knn = KNN(n=1, min_count=10)
neighbors = knn.get_nearest_neighbors(tokens_by_pitch)

In [4]:
ngram_model = Model(loader=loader, neighbors=neighbors, n=3)
ngram_model.note_table_train()

In [5]:
BOS = ("<BOS>", "<BOS>", "<BOS>", "<BOS>")
EOS = ("<EOF>", "<EOF>", "<EOF>", "<EOF>")

In [6]:
next_token = None
tokens = (BOS, BOS)

while next_token != EOS:
    next_token = ngram_model.note_table_predict(tokens)[0]
    tokens += (next_token,)

In [7]:
len(tokens)

3796

In [8]:
len(batch[0]["pitch"])

2274

In [9]:
from data_processing.music_representations.decoders import DecodeContext, create_decoder
from data_processing.music_representations.helpers.adapters import canonical_frames_to_midi

# Generated tuples -> canonical -> MIDI. Nothing goes straight to MIDI: canonical is
# the only thing that writes a .mid, so this is the same path the built dataset takes.
decoder = create_decoder("note_table", dataloader.config, dataloader.storage)
frames = decoder.decode_tuples(tokens, DecodeContext(piece_id="ngram_sample"))
frames["notes"].head()

piece_id,track_id,note_id,onset_tick,duration_tick,onset_quarter,duration_quarter,onset_sec,duration_sec,pitch,velocity
str,i16,i32,i64,i64,f64,f64,f64,f64,u8,u8
"""ngram_sample""",0,0,0,540,0.0,1.125,0.0,0.5625,47,36
"""ngram_sample""",0,1,180,600,0.375,1.25,0.1875,0.625,35,52
"""ngram_sample""",0,2,360,240,0.75,0.5,0.375,0.25,85,52
"""ngram_sample""",0,3,600,240,1.25,0.5,0.625,0.25,86,52
"""ngram_sample""",0,4,840,300,1.75,0.625,0.875,0.3125,84,44


In [10]:
midi_path = canonical_frames_to_midi(frames, Path("../generated/ngram_sample.mid"))
print(midi_path, frames["notes"].height, "notes")

# What this rendering is allowed to have lost, per the representation's own contract.
for line in decoder.tuple_contract().describe_loss():
    print(" -", line)

../generated/ngram_sample.mid 3793 notes
 - note onsets are quantized to within 0.0625 quarters
 - note durations are quantized to within 0.0625 quarters
 - note velocities are binned to within +/-4
 - tempo changes are stored only in the canonical source tables
 - time signatures are stored only in the canonical source tables
 - key signatures are stored only in the canonical source tables
 - control changes are stored only in the canonical source tables
 - pitch bends are stored only in the canonical source tables
 - track identity (program, drum flag, name) is not recoverable
 - only relative timing survives; the decoded piece always starts at tick 0
 - note_id is regenerated on decode and carries no source meaning
 - onsets are rebuilt from independently rounded deltas, so timing error accumulates along the sequence
